# Feasible operating regions with nested limit contours

**Shows:** one panel per case. The dashed outline is the region reachable with perfect coordination,
the solid outline the region without it. Inside the solid region, nested contours show how far the
operating point can move for each limit; the cross marks the operating point.

**Graph type:** regions and contours on a plane (`lf.region`, `lf.contours`, `lf.level_key`, `lf.grid`).

**Writes:** `feasible_regions.svg` next to this notebook.

**To tweak:** change the capitalised settings at the top of the **Figure** cell and re-run that cell.

In [ ]:
import numpy as np

import labfigs as lf

lf.use()

## Data

Synthetic; swap this cell for your own loading.

- `p`, `q`: grids of active and reactive power
- `cases`: one entry per panel, with
  - `outer`, `inner`: corner points of the two regions
  - `point`: the operating point
  - `unbalance`: the quantity the contours are drawn from, on the `p`, `q` grid

In [ ]:
p, q = np.meshgrid(np.linspace(8, 33, 300), np.linspace(-2, 23, 300))
shapes = (
    ([(14.5, 19.9), (30.5, 20.3), (31.5, 19), (31.7, 3.4), (14, 2.2), (13.8, 3)],
     [(14.1, 2.3), (15.3, 16.5), (20.5, 15), (29.9, 6.3), (27.3, 4.1)], (19, 10.7), 11, (22.8, 11.3)),
    ([(11.8, 18.6), (28.3, 19.2), (29, 18.3), (29, 2), (11.6, 1), (11.3, 2)],
     [(11.5, 4.8), (12.8, 17.6), (19.5, 18.3), (27, 11.6)], (22, 8.8), 12, (19.7, 9.9)),
    ([(11, 17.9), (27.3, 18.6), (28, 17.6), (28, 1.3), (10.7, 0.5), (10.5, 1.5)],
     [(15.7, 0.8), (19.2, 15.6), (21.6, 14.9), (27.5, 9.3), (27.1, 3.2), (25, 2)], (20, 13.3), 11.5, (19, 9)),
)
cases = [
    {"outer": outer, "inner": inner, "point": point,
     "unbalance": np.hypot(p - centre[0], (q - centre[1]) / 1.2) / reach}
    for outer, inner, centre, reach, point in shapes
]

## Figure

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width, including the legend above it
- `COLOURS`: colour of each panel's regions; the contours run from `CONTOUR_START` to that colour
- `LIMITS`: contour levels, lowest first
- `KEY_TITLE`: title of the key that lists the levels
- `OUTER_LABEL`, `INNER_LABEL`: legend entries of the two regions
- `LETTERS`: letter under each panel
- `X_LABEL`, `X_LIMITS`, `X_TICKS` and `Y_LABEL`, `Y_LIMITS`, `Y_TICKS`: the axes

In [ ]:
NAME = "feasible_regions"
PANEL_ASPECT = 1.45
COLOURS, CONTOUR_START = ("C2", "C3", "C4"), lf.CRIMSON
LIMITS = np.arange(0.1, 1.01, 0.1)
KEY_TITLE = r"VUF limit (\%):"
OUTER_LABEL, INNER_LABEL = "Perfect coordination", "No coordination"
LETTERS = "abc"
X_LABEL, X_LIMITS, X_TICKS = "$P$ (kW)", (8, 33), np.arange(10, 31, 5)
Y_LABEL, Y_LIMITS, Y_TICKS = "$Q$ (kVAr)", (-2, 23), np.arange(0, 21, 5)

fig, axes = lf.subplots(1, len(cases), aspect=PANEL_ASPECT, sharey=True)
for ax, letter, color, case in zip(axes, LETTERS, COLOURS, cases):
    shades = lf.ramp(len(LIMITS), (CONTOUR_START, color))
    lf.region(ax, case["outer"], color, dashed=True, label=OUTER_LABEL)
    free = lf.region(ax, case["inner"], color, label=INNER_LABEL)
    lf.contours(ax, p, q, case["unbalance"], LIMITS, shades, within=free)
    ax.plot(*case["point"], "+", color=lf.TRUE)
    lf.level_key(ax, [f"{v:.1f}" for v in LIMITS[::-1]], shades[::-1], KEY_TITLE)
    lf.legend_above(ax, gap=0.24, ncols=1, frameon=False)
    lf.grid(ax)
    ax.set(aspect="equal", xlim=X_LIMITS, ylim=Y_LIMITS, xticks=X_TICKS, yticks=Y_TICKS)
    ax.set_xlabel(f"{X_LABEL}\n({letter})")
axes[0].set_ylabel(Y_LABEL)
lf.save(fig, NAME, trim=False)